# 第 4 章习题与解答

> 本章习题聚焦注意力的**手算**和**工程细节**。建议先自己想,再展开答案。

## Exercise 4.1(易)

**题目**:给定 3 个向量,手算 naive self-attention(不加权重、不加 mask)的输出。

$$Q = K = V = \begin{bmatrix} 1 & 0 \\ 0 & 1 \\ 1 & 1 \end{bmatrix}, \quad d_k = 2$$

计算 $\text{softmax}(QK^T / \sqrt{d_k}) \cdot V$ 的前两行。

<details><summary><b>参考答案</b></summary>

**Step 1: 计算 $QK^T$(3×2 · 2×3 = 3×3)**

$$QK^T = \begin{bmatrix} 1 & 0 \\ 0 & 1 \\ 1 & 1 \end{bmatrix} \begin{bmatrix} 1 & 0 & 1 \\ 0 & 1 & 1 \end{bmatrix} = \begin{bmatrix} 1 & 0 & 1 \\ 0 & 1 & 1 \\ 1 & 1 & 2 \end{bmatrix}$$

**Step 2: 除以 $\sqrt{d_k} = \sqrt{2} \approx 1.414$**

$$\text{scores} = \begin{bmatrix} 0.707 & 0 & 0.707 \\ 0 & 0.707 & 0.707 \\ 0.707 & 0.707 & 1.414 \end{bmatrix}$$

**Step 3: softmax(每行)**

第 0 行:$e^{0.707}, e^0, e^{0.707} \approx 2.03, 1.00, 2.03$。总和 $5.06$。

$$\text{attn}_0 = \left[\frac{2.03}{5.06}, \frac{1}{5.06}, \frac{2.03}{5.06}\right] \approx [0.40, 0.20, 0.40]$$

第 1 行同理:$[0.20, 0.40, 0.40]$

**Step 4: 加权求和(attn · V)**

第 0 行输出:$0.40 \cdot [1,0] + 0.20 \cdot [0,1] + 0.40 \cdot [1,1] = [0.80, 0.60]$

第 1 行输出:$0.20 \cdot [1,0] + 0.40 \cdot [0,1] + 0.40 \cdot [1,1] = [0.60, 0.80]$

**观察**:token 0 和 token 1 对彼此的关注度(0.20)低于对 token 2 的(0.40),因为 token 2 $= [1,1]$ 和两者都相似。这就是注意力的直觉 —— **相似度高的词获得更多关注**。

</details>

In [ ]:
# 代码验证 Exercise 4.1
import torch
import torch.nn.functional as F

QKV = torch.tensor([[1., 0.], [0., 1.], [1., 1.]])
d_k = 2
scores = QKV @ QKV.T / (d_k ** 0.5)
attn = F.softmax(scores, dim=-1)
out = attn @ QKV
print("scores:\n", scores)
print("attn weights:\n", attn)
print("output:\n", out)
# 应该和手算结果一致

## Exercise 4.2(中)

**题目**:`repeat_kv` 函数(`model_minimind.py:86-89`)用了 `expand` + `reshape`,而不是 `repeat` 或 `cat`。为什么?如果改用 `torch.repeat` 会怎样?

<details><summary><b>参考答案</b></summary>

```python
# minimind 的写法(line 89):
x[:, :, :, None, :].expand(bs, slen, n_kv, n_rep, d_h).reshape(bs, slen, n_kv * n_rep, d_h)
```

关键在于 `expand` 和 `repeat` 的区别:

| 操作 | 是否复制数据 | 内存占用 | 修改副本是否影响原数据 |
|---|---|---|---|
| `expand` | **否**,只改 stride | 零额外内存 | 是(共享存储) |
| `repeat` | **是**,真复制 | $n_{rep}$ 倍 | 否 |
| `cat` | 是 | $n_{rep}$ 倍 | 否 |

`repeat_kv` 在推理时被调用,如果用 `repeat`,每一步生成都会复制一份 KV,在长序列(32768)下会浪费大量显存。`expand` 创建的是**视图(view)**,物理上只有一份数据。

**但注意**:`expand` 后跟 `reshape`,如果 `reshape` 需要内存连续,PyTorch 会隐式调用 `contiguous()` 做一次拷贝。minimind 这里之所以安全,是因为后续的 `transpose(1,2)` 和 `@` 都能处理非连续张量。

> `expand` 之所以能做「逻辑复制」,是因为它复制的维度 stride=0 —— 不管索引怎么变,都指向同一块数据。这是一个精巧的 PyTorch 底层技巧。

</details>

In [ ]:
# 验证 Exercise 4.2: expand vs repeat
x = torch.randn(1, 1, 4, 96)  # (b, T, n_kv, d_h)

# expand 方式(minimind)
expanded = x[:, :, :, None, :].expand(1, 1, 4, 2, 96).reshape(1, 1, 8, 96)
print(f"expand 后 shape: {expanded.shape}")   # (1, 1, 8, 96)
print(f"expand 的 storage 大小: {expanded.storage().nbytes()} bytes")

# repeat 方式
repeated = x.repeat(1, 1, 2, 1)  # 在 n_kv 维重复 2 次
print(f"repeat 后 shape: {repeated.shape}")   # (1, 1, 8, 96)
print(f"repeat 的 storage 大小: {repeated.storage().nbytes()} bytes")

# 两者结果相同
print(f"结果相同: {(expanded == repeated).all().item()}")

## Exercise 4.3(难)

**题目**:如果把 `q_norm` 和 `k_norm`(`model_minimind.py:104-105`)去掉,即不做 QK-Norm,训练时会发生什么?从**梯度爆炸**的角度分析。

<details><summary><b>参考答案</b></summary>

去掉 QK-Norm 后,`Q @ K^T` 的数值范围取决于 Q/K 的 magnitude。训练中,Wq/Wk 的权重会被梯度更新,如果某些维度持续增大:

1. **scores 爆炸**:Q 的某些维度变大 → $QK^T$ 的某些项变大 → scores 远超正常范围
2. **softmax 饱和**:大 scores → softmax 变成 near-one-hot → 某个 token 独占全部注意力
3. **梯度消失**:softmax 饱和时,梯度趋近于 0(因为 $\frac{\partial \text{softmax}}{\partial s_i} = p_i(1-p_i)$,当 $p_i \to 1$ 或 $p_i \to 0$ 时梯度都趋于 0)
4. **恶性循环**:梯度消失 → Q/K 无法被有效更新 → 数值继续偏大 → 更严重的饱和

**数值验证**:

$$\text{Var}(QK^T) = d_h \cdot \text{Var}(Q) \cdot \text{Var}(K)$$

如果 Q/K 的每个维度方差 = $\sigma^2$,scores 的方差 = $d_h \cdot \sigma^4$。$d_h = 96$ 时,即使 $\sigma = 2$,scores 标准差也达到 $\sqrt{96 \times 16} \approx 39$。而 softmax 在 |scores| > 5 时就接近饱和了。

**$\sqrt{d_k}$ 缩放**只处理了维度增长导致的方差增长,**没有处理权重本身变大**的问题。QK-Norm 通过归一化 Q/K 的每个 head_dim,把 $\text{Var}(Q)$ 和 $\text{Var}(K)$ 固定在 1 附近,从根源上防止了 scores 爆炸。

> 这也是为什么 QK-Norm 在现代 LLM(Qwen3、Gemma 2、DeepSeek-V2)中被广泛采用 —— 它是稳定大模型训练的一个关键 trick。

</details>

In [ ]:
# 验证 Exercise 4.3: 有/无 QK-Norm 的 scores 分布
import torch
import torch.nn.functional as F

d_h = 96
torch.manual_seed(0)

# 模拟训练后期 Wq/Wk 权重变大
Q = torch.randn(1, 8, 100, d_h) * 3   # 故意放大
K = torch.randn(1, 8, 100, d_h) * 3

# 无 QK-Norm
scores_raw = (Q @ K.transpose(-2, -1)) / (d_h ** 0.5)
attn_raw = F.softmax(scores_raw, dim=-1)
print(f"无 QK-Norm:")
print(f"  scores 标准差: {scores_raw.std():.2f}")
print(f"  最大注意力权重: {attn_raw.max():.4f}  (接近 1 = 饱和!)")
print(f"  熵: {(-attn_raw * attn_raw.clamp(min=1e-9).log()).sum(-1).mean():.4f}  (低 = 尖锐)")

# 有 QK-Norm
weight = torch.ones(d_h)
Q_n = Q * torch.rsqrt(Q.pow(2).mean(-1, keepdim=True) + 1e-6)
K_n = K * torch.rsqrt(K.pow(2).mean(-1, keepdim=True) + 1e-6)
scores_n = (Q_n @ K_n.transpose(-2, -1)) / (d_h ** 0.5)
attn_n = F.softmax(scores_n, dim=-1)
print(f"\n有 QK-Norm:")
print(f"  scores 标准差: {scores_n.std():.2f}")
print(f"  最大注意力权重: {attn_n.max():.4f}  (更均匀)")
print(f"  熵: {(-attn_n * attn_n.clamp(min=1e-9).log()).sum(-1).mean():.4f}  (高 = 平滑)")